# 기후 변화 피해국 EDA Tutorial
## Kaggle API로 실제 데이터를 받아서 분석하는 완전 가이드

**분석 목표:**
- 기후 변화로 가장 큰 피해를 입는 국가들의 공통된 특성 파악
- 경제적 취약성과 기후 피해의 상관관계 규명
- 자연재해 피해의 시계열 트렌드 분석

**사용 데이터셋 (Kaggle):**

| # | 데이터셋 | 원본 출처 | 내용 |
|---|----------|-----------|------|
| 1 | `brsdincer/all-natural-disasters-19002021-eosdis` | EM-DAT / CRED | 1900~2021 전세계 자연재해 이벤트 |
| 2 | `berkeleyearth/climate-change-earth-surface-temperature-data` | NASA / Berkeley Earth | 지표면 온도 이상치 시계열 |
| 3 | `kaggle/world-development-indicators` | World Bank | GDP, 인구, 농업 의존도 등 국가별 지표 |
| 4 | `thedevastator/nd-gain-country-index` | Notre Dame ND-GAIN | 기후 취약성 + 적응역량 국가 점수 |

**분석 흐름:**
```
[Kaggle API] → [Raw Data] → [전처리/병합] → [EDA 시각화] → [인사이트]
```

---
## Part 0 — 환경 설정 & Kaggle API 인증

### Step 0-1. 필요 패키지 설치

아래 셀을 처음 한 번만 실행하세요.

In [3]:
%pip install -q kaggle pandas numpy matplotlib seaborn plotly geopandas

Note: you may need to restart the kernel to use updated packages.


### Step 0-2. Kaggle API 키 설정

Kaggle API를 사용하려면 `kaggle.json` 파일이 필요합니다.

**발급 방법:**
1. [kaggle.com](https://www.kaggle.com) 로그인
2. 우측 상단 프로필 → **Settings**
3. **API** 섹션 → **Create New Token** 클릭
4. 다운로드된 `kaggle.json` 파일을 아래 경로에 복사:
   - **Windows:** `C:\Users\<사용자명>\.kaggle\kaggle.json`
   - **Mac/Linux:** `~/.kaggle/kaggle.json`

```json
// kaggle.json 내용 예시
{"username":"your_username","key":"xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"}
```

>  `kaggle.json`은 절대 GitHub에 올리면안된다. `.gitignore`에 추가

In [4]:
import os
import json

# 너의 Local Computer 안에 있는 Path 경로를 확인할수 있는 Class
from pathlib import Path

# kaggle.json 위치 확인
kaggle_path = Path.home() / ".kaggle" / "kaggle.json"
print(kaggle_path)

if kaggle_path.exists():
    print(f"kaggle.json 발견: {kaggle_path}")
    with open(kaggle_path) as f:
        creds = json.load(f)
    print(f"사용자: {creds.get('username')}")
    # API키 보안: 앞 4자리만 표시
    key = creds.get('key')
    print(f"API Key: {key[:4]}{'*' * (len(key)-4)}")
else:
    print(f"kaggle.json 없음: {kaggle_path}")
    print("위의 Step 0-2 가이드를 따라 설정하세요.")

/Users/yinayoon/.kaggle/kaggle.json
kaggle.json 발견: /Users/yinayoon/.kaggle/kaggle.json
사용자: yinayoon
API Key: KGAT*********************************


---
## Part 1 — 데이터 수집: Kaggle API

### Kaggle API 사용법 기초

```bash
# CLI로 직접 다운로드할 때
kaggle datasets download -d <owner>/<dataset-name> -p ./data --unzip

# 데이터셋 검색
kaggle datasets list -s "climate change"
```

Python에서는 `kaggle.api` 객체를 사용합니다.

### Step 1-1. 다운로드 경로 준비

In [5]:
import subprocess
from pathlib import Path
import zipfile

# Kaggle API로 데이터 다운로드
def download_kaggle_dataset(dataset_name, path):
    print(f"\n Downloading: {dataset_name}")
    print(f"saving to: {path}")
    
    cmd = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        dataset_name,
        "-p",
        str(path)
    ]
    
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error downloading {dataset_name}: {e}")

# Data Unzipping
def unzip_in_place(path):
    for zip_file in path.glob("*.zip"):
        print(f"unzipping {zip_file.name}")
        
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(path)

# 데이터 저장 폴더 생성
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# 각 데이터셋별 하위 폴더
dirs = {
    "disasters": DATA_DIR / "disasters",
    "temperature": DATA_DIR / "temperature",
    "wdi": DATA_DIR / "wdi",
    "ndgain": DATA_DIR / "ndgain",
}
    
### Kagggle dataset Mapping
datasets = {
    "disasters": "brsdincer/all-natural-disasters-19002021-eosdis",
    "temperature": "berkeleyearth/climate-change-earth-surface-temperature-data",
    "wdi": "kaggle/world-development-indicators",
    "ndgain": "thedevastator/nd-gain-country-index",
}

for name, path in dirs.items():
    path.mkdir(exist_ok=True)
    print(f"{path}")
    
    # download
    download_kaggle_dataset(datasets[name], path)
    
    # unzip
    unzip_in_place(path)
    
print("\n Data Download and Unzip Completed!")

data/disasters

 Downloading: brsdincer/all-natural-disasters-19002021-eosdis
saving to: data/disasters
Dataset URL: https://www.kaggle.com/datasets/brsdincer/all-natural-disasters-19002021-eosdis
License(s): DbCL-1.0
all-natural-disasters-19002021-eosdis.zip: Skipping, found more recently modified local copy (use --force to force download)
unzipping all-natural-disasters-19002021-eosdis.zip
data/temperature

 Downloading: berkeleyearth/climate-change-earth-surface-temperature-data
saving to: data/temperature
Dataset URL: https://www.kaggle.com/datasets/berkeleyearth/climate-change-earth-surface-temperature-data
License(s): CC-BY-NC-SA-4.0
climate-change-earth-surface-temperature-data.zip: Skipping, found more recently modified local copy (use --force to force download)
unzipping climate-change-earth-surface-temperature-data.zip
data/wdi

 Downloading: kaggle/world-development-indicators
saving to: data/wdi
Dataset URL: https://www.kaggle.com/datasets/kaggle/world-development-indicator

In [6]:
import pandas as pd

df = pd.read_csv('/Users/yinayoon/Documents/Projects/EDA-Projects/data/wdi/Country.csv')


df.head(11)

,CountryCode,ShortName,TableName,LongName,Alpha2Code,CurrencyUnit,SpecialNotes,Region,IncomeGroup,Wb2Code,...,GovernmentAccountingConcept,ImfDataDisseminationStandard,LatestPopulationCensus,LatestHouseholdSurvey,SourceOfMostRecentIncomeAndExpenditureData,VitalRegistrationComplete,LatestAgriculturalCensus,LatestIndustrialData,LatestTradeData,LatestWaterWithdrawalData
0,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,AF,...,Consolidated central government,General Data Dissemination System (GDDS),1979,"Multiple Indicator Cluster Survey (MICS), 2010/11","Integrated household survey (IHS), 2008",NaN,2013/14,NaN,2013.0,2000.0
1,ALB,Albania,Albania,Republic of Albania,AL,Albanian lek,NaN,Europe & Central Asia,Upper middle income,AL,...,Budgetary central government,General Data Dissemination System (GDDS),2011,"Demographic and Health Survey (DHS), 2008/09",Living Standards Measurement Study Survey (LSM...,Yes,2012,2011.0,2013.0,2006.0
2,DZA,Algeria,Algeria,People's Democratic Republic of Algeria,DZ,Algerian dinar,NaN,Middle East & North Africa,Upper middle income,DZ,...,Budgetary central government,General Data Dissemination System (GDDS),2008,"Multiple Indicator Cluster Survey (MICS), 2012","Integrated household survey (IHS), 1995",NaN,NaN,2010.0,2013.0,2001.0
3,ASM,American Samoa,American Samoa,American Samoa,AS,U.S. dollar,NaN,East Asia & Pacific,Upper middle income,AS,...,NaN,NaN,2010,NaN,NaN,Yes,2007,NaN,NaN,NaN
4,ADO,Andorra,Andorra,Principality of Andorra,AD,Euro,NaN,Europe & Central Asia,High income: nonOECD,AD,...,NaN,NaN,2011. Population data compiled from administra...,NaN,NaN,Yes,NaN,NaN,2006.0,NaN
5,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Upper middle income,AO,...,Budgetary central government,General Data Dissemination System (GDDS),2014,"Malaria Indicator Survey (MIS), 2011","Integrated household survey (IHS), 2008/09",NaN,2015,NaN,NaN,2005.0
6,ATG,Antigua and Barbuda,Antigua and Barbuda,Antigua and Barbuda,AG,East Caribbean dollar,April 2012 database update: Based on official ...,Latin America & Caribbean,High income: nonOECD,AG,...,Budgetary central government,General Data Dissemination System (GDDS),2011,NaN,NaN,Yes,2007,NaN,2013.0,2005.0
7,ARB,Arab World,Arab World,Arab World,1A,NaN,Arab World aggregate. Arab World is composed o...,NaN,NaN,1A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ARG,Argentina,Argentina,Argentine Republic,AR,Argentine peso,The base year has changed to 2004.,Latin America & Caribbean,High income: nonOECD,AR,...,Consolidated central government,Special Data Dissemination Standard (SDDS),2010,"Multiple Indicator Cluster Survey (MICS), 2011/12","Integrated household survey (IHS), 2012",Yes,2013,2002.0,2013.0,2011.0
9,ARM,Armenia,Armenia,Republic of Armenia,AM,Armenian dram,NaN,Europe & Central Asia,Lower middle income,AM,...,Consolidated central government,Special Data Dissemination Standard (SDDS),2011,"Demographic and Health Survey (DHS), 2010","Integrated household survey (IHS), 2012",Yes,2013/14,2008.0,2013.0,2012.0


In [7]:
import os
print(os.getcwd())

/Users/yinayoon/Documents/Projects/EDA-Projects


In [8]:
data = df.dropna()
print(data)
#why empty dataframe..

Empty DataFrame
Columns: [CountryCode, ShortName, TableName, LongName, Alpha2Code, CurrencyUnit, SpecialNotes, Region, IncomeGroup, Wb2Code, NationalAccountsBaseYear, NationalAccountsReferenceYear, SnaPriceValuation, LendingCategory, OtherGroups, SystemOfNationalAccounts, AlternativeConversionFactor, PppSurveyYear, BalanceOfPaymentsManualInUse, ExternalDebtReportingStatus, SystemOfTrade, GovernmentAccountingConcept, ImfDataDisseminationStandard, LatestPopulationCensus, LatestHouseholdSurvey, SourceOfMostRecentIncomeAndExpenditureData, VitalRegistrationComplete, LatestAgriculturalCensus, LatestIndustrialData, LatestTradeData, LatestWaterWithdrawalData]
Index: []

[0 rows x 31 columns]


In [9]:
col_w_most_nans = df.isnull().sum().idxmax()
print(col_w_most_nans)

AlternativeConversionFactor


In [15]:
nancols = df.columns[df.isna().any()]
print(nancols)

for col in nancols:
    df = df.drop(columns=col)
df

Index([], dtype='object')


,CountryCode,ShortName,TableName,LongName
0,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan
1,ALB,Albania,Albania,Republic of Albania
2,DZA,Algeria,Algeria,People's Democratic Republic of Algeria
3,ASM,American Samoa,American Samoa,American Samoa
4,ADO,Andorra,Andorra,Principality of Andorra
...,...,...,...,...
242,WBG,West Bank and Gaza,West Bank and Gaza,West Bank and Gaza
243,WLD,World,World,World
244,YEM,Yemen,"Yemen, Rep.",Republic of Yemen
245,ZMB,Zambia,Zambia,Republic of Zambia


In [ ]:
threshold = 0.7
df = df.loc[:, df.isna().mean() < threshold]
df

In [29]:
df1 = pd.read_csv('/Users/yinayoon/Documents/Projects/EDA-Projects/data/temperature/GlobalTemperatures.csv')

# Clean and prepare temperature data
df1 = df1.dropna()
X = df1.select_dtypes(include=[np.number]).drop(['LandAverageTemperature'], axis=1)  # Use temperature as target
y = df1['LandAverageTemperature']

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=8, test_size=0.3)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train decision tree for feature selection
regressor = DecisionTreeRegressor(random_state=8, max_depth=6)
regressor.fit(X_train_scaled, y_train)

# Get feature importances
importances = regressor.feature_importances_

# Select features with importance greater than a threshold
threshold = 0.1  # Adjust as needed
selected_features = X.columns[importances > threshold]

print(f"Selected {len(selected_features)} features out of {len(X.columns)} total features:")
print("Selected features:", list(selected_features))

# Show feature importance ranking
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 10 most important features:")
print(feature_importance_df.head(10))

# Use only the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

NameError: name 'MinMaxScaler' is not defined

In [27]:
# Decision Tree Feature Selection for Indicators.csv
df1 = pd.read_csv('/Users/yinayoon/Documents/Projects/EDA-Projects/data/wdi/Indicators.csv')

# Clean and prepare indicators data
df1 = df1.dropna()
X = df1.select_dtypes(include=[np.number]).drop(['Value'], axis=1)  # Use Value as target
y = df1['Value']

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=8, test_size=0.3)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train decision tree for feature selection
regressor = DecisionTreeRegressor(random_state=8, max_depth=6)
regressor.fit(X_train_scaled, y_train)

# Get feature importances
importances = regressor.feature_importances_

# Select features with importance greater than a threshold
threshold = 0.1  # Adjust as needed
selected_features = X.columns[importances > threshold]

print(f"Selected {len(selected_features)} features out of {len(X.columns)} total features:")
print("Selected features:", list(selected_features))

# Show feature importance ranking
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 10 most important features:")
print(feature_importance_df.head(10))

# Use only the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

Selected 1 features out of 1 total features:
Selected features: ['Year']

Top 10 most important features:
  Feature  Importance
0    Year         1.0


In [26]:
# Decision Tree Feature Selection for Disasters Data
from pathlib import Path

# Find the disaster CSV file
disaster_path = next(Path('/Users/yinayoon/Documents/Projects/EDA-Projects/data/disasters').rglob('1970-2021_DISASTERS*.csv'), None)
if disaster_path is None:
    raise FileNotFoundError('Could not find the 1970-2021 disaster CSV file')

print(f'Loading disaster file: {disaster_path}')
df1 = pd.read_csv(disaster_path)

print(f"Initial dataset shape: {df1.shape}")
print(f"Columns with NaN: {df1.columns[df1.isna().any()].tolist()}")

# Instead of dropping all NaN rows, fill NaN with 0 for numeric columns
# This preserves more data for analysis
numeric_cols = df1.select_dtypes(include=[np.number]).columns
df1[numeric_cols] = df1[numeric_cols].fillna(0)

print(f"After filling NaN with 0: {df1.shape}")

# Choose a target variable
if 'Total Deaths' in df1.columns:
    target_col = 'Total Deaths'
elif 'Total Damages' in df1.columns:
    target_col = 'Total Damages'
else:
    target_col = numeric_cols[0]

print(f"Using '{target_col}' as target variable")

X = df1.select_dtypes(include=[np.number]).drop([target_col], axis=1, errors='ignore')
y = df1[target_col]

print(f"Features shape: {X.shape}, Target shape: {y.shape}")

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=8, test_size=0.3)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train decision tree for feature selection
regressor = DecisionTreeRegressor(random_state=8, max_depth=6)
regressor.fit(X_train_scaled, y_train)

# Get feature importances
importances = regressor.feature_importances_

# Select features with importance greater than a threshold
threshold = 0.01  # Lower threshold for disasters data
selected_features = X.columns[importances > threshold]

print(f"Selected {len(selected_features)} features out of {len(X.columns)} total features:")
print("Selected features:", list(selected_features))

# Show feature importance ranking
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nTop 10 most important features:")
print(feature_importance_df.head(10))

# Use only the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

Loading disaster file: /Users/yinayoon/Documents/Projects/EDA-Projects/data/disasters/DISASTERS/1970-2021_DISASTERS.xlsx - emdat data.csv
Initial dataset shape: (14644, 47)
Columns with NaN: ['Glide', 'Disaster Subtype', 'Disaster Subsubtype', 'Event Name', 'Location', 'Origin', 'Associated Dis', 'Associated Dis2', 'OFDA Response', 'Appeal', 'Declaration', 'Aid Contribution', 'Dis Mag Value', 'Dis Mag Scale', 'Latitude', 'Longitude', 'Local Time', 'River Basin', 'Start Month', 'Start Day', 'End Month', 'End Day', 'Total Deaths', 'No Injured', 'No Affected', 'No Homeless', 'Total Affected', "Reconstruction Costs ('000 US$)", "Insured Damages ('000 US$)", "Total Damages ('000 US$)", 'CPI', 'Adm Level', 'Admin1 Code', 'Admin2 Code', 'Geo Locations']
After filling NaN with 0: (14644, 47)
Using 'Total Deaths' as target variable
Features shape: (14644, 18), Target shape: (14644,)
Selected 10 features out of 18 total features:
Selected features: ['Year', 'Aid Contribution', 'Start Year', 'End